# Policy Gradient

---
## 目的
Policy Gradient（方策勾配法）の仕組みを理解し，ゲームタスクを用いて強化学習を行う．
学習後のエージェントの可視化を行い，学習がうまくできているか確認を行う．

## Policy Gradient
Policy Gradientは強化学習の1手法で，Deep Q-Networkなどの行動価値関数（Q値）を用いて行動を決定し，最適な行動価値関数を求めていく手法とは違い，方策を用いた予測により行動の決定を行い，行動により得られた報酬をもとに方策を直接改善する手法です．Policy Gradientは方策をもとに行動を決定し，改善していく手法のため，方策ベースと呼ばれています．方策勾配法は以下の方策勾配定理をもとに勾配を計算していきます．（証明略）

$$
\nabla_\theta J=\mathbb{E}\left[\nabla_\theta \log\pi_\theta(a|s)Q^{\pi}(s,a)\right]
$$

方策ベースの手法は，状態に対する行動の確率分布を直接出力するため，連続的な行動空間での学習を行うことが可能であり，主にロボットタスクなどに使用されています．派生手法には，TRPO，PPOなどがあります．

今回は，最も基礎的な方策勾配法であるREINFORCEアルゴリズムを実装します．REINFORCEでは，1エピソード分の行動とその際の対数確率（log probability）を記録しておき，エピソード終了後に実際に得られた割引累積報酬（収益）$G_t$を$Q^{\pi}(s,a)$の代わりに用いて，以下の損失関数を最小化するように方策を更新します．

$$
L=-\sum_{t}\log\pi_\theta(a_t|s_t)G_t
$$


## 準備
下記のプログラムを実行して，実験に必要な追加ライブラリをインストールする．

In [ ]:
!pip install -q "gymnasium[atari,other]" ale-py

## モジュールのインポートとGPUの確認
はじめに必要なモジュールをインポートする．

今回はPyTorchに加えて，Pongを実行するためのシミュレータであるGymnasium（gymnasium）をインポートする．
そして，GPUが使用可能かどうかを確認する．

In [ ]:
import time
import datetime
import random
import collections
import cv2
from base64 import b64encode

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Use device:', device)

## シード値の固定

In [ ]:
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True

## OpenAI GymによるPong環境の定義
[Gymnasium](https://gymnasium.farama.org/)は，様々な種類の環境を提供しているモジュールです．
今回は，Gymnasiumで利用可能なAtari2600のゲームであるPongを使用します．
Pongの行動数は6ですが，実質的な行動はパドルを上下どちらかに移動させる2種類のみです（詳細は`deep_q_network.ipynb`を参照）．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

obs, info = env.reset(seed=seed, options=None)
print('observation space:', env.observation_space)
print('action space:', env.action_space)
print('initial observation:', obs.shape)

## 環境の前処理
Atari環境の学習を安定・効率化するため，`deep_q_network.ipynb`と同様の前処理を適用します．

* MaxAndSkipEnv：1ステップ実行毎に，4フレームゲームを進める（skip frame）
* FireResetEnv：エピソード（ゲーム）開始にFireを実行しなければ開始されない環境でのreset関数の設定
* ProcessFrame84：210×160のRGB画像を84×84のグレースケール画像に変換
* ImageToPyTorch：観測情報（画像）のshapeをHWC（高さ，幅，チャネル）からCHW（チャネル，高さ，幅）に変換
* ScaledFloatFrame：画像（0から255）を0.0から1.0の範囲で正規化
* BufferWrapper：観測情報を4フレームまとめて返す

In [ ]:
class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = None
        for _ in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if done:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, done, truncated, info

class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(1)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(2)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        return obs, info

class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            assert False, "Unknown resolution."
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)

class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(old_shape[-1], old_shape[0], old_shape[1]),
                                                dtype=np.float32)

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)

class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0

class BufferWrapper(gym.ObservationWrapper):
    def __init__(self, env, n_steps, dtype=np.float32):
        super(BufferWrapper, self).__init__(env)
        self.dtype = dtype
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(old_space.low.repeat(n_steps, axis=0),
                                                old_space.high.repeat(n_steps, axis=0), dtype=dtype)

    def reset(self, **kwargs):
        self.buffer = np.zeros_like(self.observation_space.low, dtype=self.dtype)
        obs, info = self.env.reset(**kwargs)
        return self.observation(obs), info

    def observation(self, observation):
        self.buffer[:-1] = self.buffer[1:]
        self.buffer[-1] = observation
        return self.buffer

### 前処理の適用
環境に対して必要となる前処理を適用します．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

env = MaxAndSkipEnv(env)
env = FireResetEnv(env)
env = ProcessFrame84(env)
env = ImageToPyTorch(env)
env = BufferWrapper(env, 4)
env = ScaledFloatFrame(env)

## ネットワーク構造
ネットワークモデルを定義します．
ここでは，環境からのゲーム画面情報（画像）を入力し，各行動を選択する確率（方策）を出力するようなネットワークを定義します．
ネットワーク構造は，畳み込み層3層と全結合層2層とします．

出力は各行動に対応するスコア（logit）とし，`torch.distributions.Categorical`を用いて確率分布へ変換します．

In [ ]:
class Policy(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(Policy, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        conv_out_size = self._get_conv_out(input_shape)
        self.fc = nn.Sequential(
            nn.Linear(conv_out_size, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def _get_conv_out(self, shape):
        o = self.conv(torch.zeros(1, *shape))
        return int(np.prod(o.size()))

    def forward(self, x):
        conv_out = self.conv(x).view(x.size()[0], -1)
        return self.fc(conv_out)

## エージェントの定義
エージェントが環境に対して方策に従い行動し，1エピソード分の行動の対数確率と報酬を収集します．

`run_episode`関数では，環境をリセットしてからエピソードが終了するまで行動を選択・実行し続け，各時刻の対数確率（`log_prob`）と報酬（`reward`）をリストに記録して返します．

In [ ]:
class Agent:
    def __init__(self, env, policy, device):
        self.env = env
        self.policy = policy
        self.device = device

    def run_episode(self):
        state, _ = self.env.reset()
        log_probs = []
        rewards = []
        done = False

        while not done:
            state_v = torch.as_tensor(state).unsqueeze(0).to(self.device)
            logits = self.policy(state_v)
            dist = Categorical(logits=logits)
            action = dist.sample()

            next_state, reward, terminated, truncated, info = self.env.step(action.item())
            done = terminated or truncated

            log_probs.append(dist.log_prob(action))
            rewards.append(reward)
            state = next_state

        return log_probs, rewards

## Lossの計算
REINFORCEでは，各時刻$t$における割引累積報酬（収益）$G_t=\sum_{k=t}^{T}\gamma^{k-t}r_k$を，末尾のステップから逆順に計算します．
$G_t$は，エピソード内やバッチ間でスケールが大きく異なるため，そのまま用いると学習が不安定になりやすくなります．そこで，平均0，分散1に正規化してから用いることで，学習を安定化させます（ベースラインによる分散削減）．

この$G_t$を用いて，以下の損失関数を計算します．

$$
L=-\sum_{t}\log\pi_\theta(a_t|s_t)G_t
$$

In [ ]:
def calc_loss(log_probs, rewards, gamma, device):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns_v = torch.tensor(returns, dtype=torch.float32, device=device)
    returns_v = (returns_v - returns_v.mean()) / (returns_v.std() + 1e-8)

    log_probs_v = torch.cat(log_probs)
    loss = -(log_probs_v * returns_v).sum()
    return loss

## 学習
Policy Gradient（REINFORCE）を用いて学習を行います．学習環境は，Atari環境のPongゲーム環境を用います．
1エピソードごとに，方策に従って行動しながら経験（対数確率と報酬）を収集し，エピソード終了後にまとめてネットワークを更新します．

Pongは報酬が疎（得点が入った時のみ±1）なQでの学習が難しい環境であり，REINFORCEは方策勾配法の中でも特にサンプル効率が低い手法のため，人間並みのスコアに到達するには非常に多くのエピソード数（数万エピソード以上）が必要です．ここでは，学習の仕組みと損失の推移を確認できる範囲のエピソード数で実行します．

In [ ]:
GAMMA = 0.99
LEARNING_RATE = 1e-4
num_episodes = 500  # 収束にはこの数十〜数百倍のエピソード数が必要

policy = Policy(env.observation_space.shape, env.action_space.n).to(device)
optimizer = optim.Adam(policy.parameters(), lr=LEARNING_RATE)
agent = Agent(env, policy, device)

record_episode = []
record_reward = []
total_rewards = []

ts = time.time()
for episode in range(1, num_episodes + 1):
    log_probs, rewards = agent.run_episode()

    loss = calc_loss(log_probs, rewards, GAMMA, device)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    episode_reward = sum(rewards)
    total_rewards.append(episode_reward)
    mean_reward = np.mean(total_rewards[-20:])
    record_episode.append(episode)
    record_reward.append(mean_reward)

    if episode % 10 == 0:
        print('Episode {0}/{1}: reward {2:.1f}, mean reward {3:.3f}, loss {4:.3f}, time {5}'.format(
            episode, num_episodes, episode_reward, mean_reward, loss.item(), datetime.timedelta(seconds=time.time() - ts)))

## 学習時のスコアの推移
横軸エピソード数，縦軸平均スコアとしたグラフを描画してみます．

In [ ]:
fig = plt.figure()
plt.plot(record_episode, record_reward, color="red")
plt.grid()
plt.xlabel("episode")
plt.ylabel("mean reward")
plt.savefig("./policy_gradient_episode_per_reward.png")
plt.show()

## 学習結果の可視化
学習したエージェントを確認してみます．`RecordVideo`ラッパーを用いて，エージェントによるゲームプレイを動画として記録し，Notebook上で再生します．評価時は，行動をサンプリングせず，最も確率の高い行動を選択します．

In [ ]:
gym.register_envs(ale_py)
eval_env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False, render_mode='rgb_array')
eval_env = MaxAndSkipEnv(eval_env)
eval_env = FireResetEnv(eval_env)
eval_env = ProcessFrame84(eval_env)
eval_env = ImageToPyTorch(eval_env)
eval_env = BufferWrapper(eval_env, 4)
eval_env = ScaledFloatFrame(eval_env)
eval_env = RecordVideo(eval_env, './video/policy_gradient', episode_trigger=lambda x: True)

state, info = eval_env.reset()
done = False
policy.eval()
with torch.no_grad():
    while not done:
        state_v = torch.as_tensor(state).unsqueeze(0).to(device)
        logits = policy(state_v)
        action = torch.argmax(logits, dim=1).item()

        state, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated

eval_env.close()

mp4 = open('./video/policy_gradient/rl-video-episode-0.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f"""
<video width="320" height="420" controls>
      <source src="{data_url}" type="video/mp4">
</video>""")

## 課題

1. 学習エピソード数`num_episodes`を増やして，学習の進み方を確認してみましょう．
2. 収益$G_t$の正規化（ベースライン）の有無で，学習の安定性がどのように変わるか比較してみましょう．
3. Pong以外のゲームで学習してみましょう．
    * `gym.make()`での環境の指定を変更することで，任意の環境で学習できます．
    * 指定できる環境は，`ALE/Breakout-v5`や`ALE/MsPacman-v5`などがあります．詳しくは[ドキュメント](https://ale.farama.org/environments/)をチェックしてください．